# Triage automatico de correo con LangChain — Nivel 2

**Proyecto Integrador — Sistemas Inteligentes 1 (Universidad de Caldas)**

Reimplementacion en **LangChain (>= 0.2)** del mismo caso de uso del **Nivel 1 (n8n)**: clasificar
correos no leidos de Gmail y, segun la categoria, redactar un **borrador** de respuesta (sin enviarlo)
o solo registrar la decision, **sin duplicar** borradores.

Este notebook esta organizado por **componentes**, en el mismo orden de la diapositiva 07:

| Seccion | Componente |
|---|---|
| 1 | Entorno y variables (.env) |
| 2 | **LLM** — ChatGroq |
| 3 | **Salida estructurada** (Pydantic) + **CoT** (campo razonamiento) |
| 4 | **ChatPromptTemplate** |
| 5 | **Chain** LCEL de clasificacion + prueba offline (4 categorias) |
| 6 | **Tools** — GmailToolkit |
| 7 | **RAG opcional** (plantillas de respuesta, desactivable) |
| 8 | Funciones auxiliares (parseo, deduplicacion, redaccion, borrador en hilo) |
| 9 | **Orquestacion** principal (equivalencia con el Switch de n8n) |
| 10 | Ejecucion sobre la bandeja real |


## 0. Requisitos previos

Antes de ejecutar, asegurate de tener en la **raiz del proyecto**:

1. **.env** con tu clave de Groq: `GROQ_API_KEY=gsk_...`
2. **credentials.json** — OAuth de escritorio descargado de Google Cloud Console (Gmail API habilitada).
   En el primer run se abrira el navegador y se generara **token.json** automaticamente.

> Detalles de credenciales en el `README.md`. Los archivos `.env`, `credentials.json` y `token.json`
> estan en `.gitignore`: **nunca se versionan**.

La siguiente celda instala dependencias (descomentala la primera vez):

In [ ]:
# Instalacion de dependencias (descomentar la primera vez).
# Recomendado: hacerlo dentro de un venv activado.
# %pip install -r requirements.txt


## 1. Entorno y variables de entorno

Cargamos las variables desde `.env` con `python-dotenv` y verificamos que exista `GROQ_API_KEY`.
**No imprimimos la clave**, solo confirmamos su presencia.

In [1]:
import os
from dotenv import load_dotenv

# Carga las variables definidas en .env hacia os.environ
load_dotenv()

# Verificacion (sin exponer el valor de la clave)
assert os.getenv("GROQ_API_KEY"), (
    "Falta GROQ_API_KEY. Copia .env.example a .env y pon tu clave de Groq."
)
print("OK - GROQ_API_KEY cargada.")


OK - GROQ_API_KEY cargada.


## 2. LLM — `ChatGroq`

Componente **LLM**. Usamos el mismo modelo y temperatura que el Nivel 1:
`llama-3.3-70b-versatile` con `temperature=0.2` (baja, para clasificaciones estables y reproducibles).

In [2]:
from langchain_groq import ChatGroq

# LLM compartido por la cadena de clasificacion y (si se usa) por la redaccion de borradores.
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # mismo modelo que el Nivel 1 (n8n)
    temperature=0.2,                  # baja -> respuestas mas deterministas
)
print("LLM listo:", llm.model_name)


LLM listo: llama-3.3-70b-versatile


## 3. Salida estructurada (Pydantic) + Cadena de pensamiento (CoT)

Componente **salida estructurada** + **CoT**. Definimos el esquema `Triage` con Pydantic. El primer campo,
`razonamiento`, implementa **Zero-shot Chain-of-Thought**: forzamos al modelo a explicar paso a paso por
que elige la categoria **antes** de comprometerse con ella. Los nombres de categoria van **sin tildes**
para coincidir exactamente con la clasificacion del Nivel 1.

In [3]:
from typing import Literal
from pydantic import BaseModel, Field


class Triage(BaseModel):
    """Salida estructurada del triage de un correo."""

    # CoT: el razonamiento va PRIMERO para que el modelo "piense antes de decidir".
    razonamiento: str = Field(
        description="Razonamiento paso a paso antes de decidir la categoria (CoT)"
    )
    categoria: Literal[
        "Urgente", "Solicitud de informacion", "Spam o promocion", "Otro"
    ] = Field(description="Categoria del correo")
    urgencia: Literal["alta", "media", "baja"] = Field(description="Nivel de urgencia")
    resumen: str = Field(description="Una sola frase con el contenido del correo")
    requiere_respuesta: bool = Field(
        description="True solo si el correo espera una respuesta del destinatario"
    )
    borrador_respuesta: str = Field(
        description=(
            "Si requiere_respuesta es True, respuesta breve, cordial y profesional en "
            "espanol; si no, cadena vacia"
        )
    )


print("Esquema Triage definido. Campos:", list(Triage.model_fields))


Esquema Triage definido. Campos: ['razonamiento', 'categoria', 'urgencia', 'resumen', 'requiere_respuesta', 'borrador_respuesta']


## 4. `ChatPromptTemplate`

Componente **prompt**. Un `system` message con las reglas de triage (reutilizado del Nivel 1) y un
`human` template que inyecta los datos del correo: remitente, asunto y cuerpo.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_TRIAGE = """Eres un asistente de triage de correo electronico. Clasifica cada correo y devuelve la respuesta en el formato estructurado solicitado.

Categorias posibles (campo categoria):
- Urgente: requiere accion o respuesta inmediata.
- Solicitud de informacion: pide datos, aclaraciones o documentos.
- Spam o promocion: publicidad, newsletters no solicitados o phishing.
- Otro: cualquier correo que no encaje en lo anterior.

Reglas:
- urgencia debe ser alta, media o baja.
- requiere_respuesta es true solo si el correo espera una respuesta del destinatario.
- borrador_respuesta: si requiere_respuesta es true, redacta una respuesta breve, cordial y profesional en espanol; si es false, deja la cadena vacia.
- resumen: una sola frase con el contenido del correo.
- razonamiento: explica brevemente por que elegiste la categoria antes de decidir."""

prompt_triage = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_TRIAGE),
    ("human",
     "Clasifica el siguiente correo:\n\n"
     "De: {remitente}\n"
     "Asunto: {asunto}\n"
     "Cuerpo: {cuerpo}"),
])

print("Prompt listo. Variables esperadas:", prompt_triage.input_variables)


Prompt listo. Variables esperadas: ['asunto', 'cuerpo', 'remitente']


## 5. Chain LCEL de clasificacion

Componente **Chain**. Componemos con LCEL el prompt con el LLM en modo **salida estructurada**:
`prompt | llm.with_structured_output(Triage)`. El resultado de `invoke(...)` es directamente un objeto
`Triage` validado por Pydantic.

In [5]:
# Cadena de clasificacion: prompt -> LLM con salida estructurada -> objeto Triage
cadena_clasificacion = prompt_triage | llm.with_structured_output(Triage)

print("Cadena de clasificacion construida (prompt | llm.with_structured_output).")


Cadena de clasificacion construida (prompt | llm.with_structured_output).


### 5.1 Prueba offline de clasificacion (4 categorias)

Validamos la cadena **sin tocar Gmail** con un correo de ejemplo por cada categoria. Esto cubre el
criterio de aceptacion "clasifica correctamente en las 4 categorias" y es reproducible para el video.

In [ ]:
correos_demo = [
    {  # -> Urgente
        "remitente": "jefe@empresa.com",
        "asunto": "URGENTE: caida del servidor de produccion",
        "cuerpo": "El sitio esta caido desde hace 10 minutos. Necesito que lo revises ya y me confirmes.",
    },
    {  # -> Solicitud de informacion
        "remitente": "cliente@correo.com",
        "asunto": "Consulta sobre la factura 0234",
        "cuerpo": "Buenas tardes, podrian enviarme el desglose de la factura 0234? Gracias.",
    },
    {  # -> Spam o promocion
        "remitente": "ofertas@promos.com",
        "asunto": "70% de descuento solo HOY, compra ya",
        "cuerpo": "Aprovecha esta oferta unica. Haz clic aqui para ganar un premio.",
    },
    {  # -> Otro
        "remitente": "boletin@universidad.edu",
        "asunto": "Boletin mensual de la facultad",
        "cuerpo": "Resumen de actividades del mes. No requiere ninguna accion de tu parte.",
    },
]

for c in correos_demo:
    t = cadena_clasificacion.invoke(c)
    print("=" * 70)
    print("Asunto:        ", c["asunto"])
    print("Categoria:     ", t.categoria, "| urgencia:", t.urgencia)
    print("Requiere resp.:", t.requiere_respuesta)
    print("Resumen:       ", t.resumen)
    print("Razonamiento:  ", t.razonamiento)
    if t.borrador_respuesta:
        print("Borrador:      ", t.borrador_respuesta[:200], "...")


## 6. Tools — `GmailToolkit`

Componente **Tools**. Construimos las credenciales OAuth y el `GmailToolkit`, que expone las tools de
Gmail. Guardamos cada tool por nombre para usarlas despues. El `api_resource` (cliente de la Gmail API
que el propio toolkit construye) lo reutilizaremos para lo que las tools no exponen: leer las etiquetas
del hilo (deduplicacion) y crear el borrador **dentro** del hilo.

Nombres de las tools del toolkit:
`search_gmail`, `get_gmail_message`, `get_gmail_thread`, `create_gmail_draft`, `send_gmail_message`.

In [ ]:
from langchain_google_community import GmailToolkit
from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)

# OAuth de escritorio: usa credentials.json y crea/usa token.json.
# Scope https://mail.google.com/ -> lectura + creacion de borradores.
credentials = get_gmail_credentials(
    token_file="token.json",
    scopes=["https://mail.google.com/"],
    client_secrets_file="credentials.json",
)
api_resource = build_resource_service(credentials=credentials)

toolkit = GmailToolkit(api_resource=api_resource)

# Diccionario {nombre_tool: tool} para acceso comodo.
tools = {t.name: t for t in toolkit.get_tools()}
print("Tools disponibles:", list(tools))

# Referencias directas a las tools que usaremos.
tool_buscar = tools["search_gmail"]          # GmailSearch
tool_hilo = tools["get_gmail_thread"]         # GmailGetThread
tool_borrador = tools["create_gmail_draft"]   # GmailCreateDraft


## 7. RAG opcional — plantillas de respuesta

Componente **RAG** (suma en la rubrica). Vector store local (**Chroma**) con plantillas de respuesta y
embeddings **gratuitos** (`sentence-transformers/all-MiniLM-L6-v2` via HuggingFace). Al redactar un
borrador, recuperamos la plantilla mas parecida para guiar tono y estructura.

- Esta **desactivado por defecto** (`USE_RAG = False`) para que el notebook corra sin descargar el modelo
  de embeddings. Para activarlo: descomenta el bloque RAG de `requirements.txt`, instala, y pon
  `USE_RAG = True`.
- Si las dependencias no estan instaladas, el codigo lo detecta y continua sin RAG (sin romper nada).

In [ ]:
USE_RAG = False  # cambia a True para activar la recuperacion de plantillas

# Plantillas de respuesta (corpus minimo del vector store).
PLANTILLAS = [
    "Estimado/a, gracias por su mensaje. Atenderemos su caso con prioridad y le confirmaremos "
    "la solucion a la mayor brevedad. Quedamos atentos.",
    "Hola, gracias por su consulta. Le adjuntamos/enviamos la informacion solicitada. Si necesita "
    "algun dato adicional, con gusto se lo facilitamos.",
    "Buenas, agradecemos su correo. Hemos registrado su solicitud y le daremos respuesta dentro de "
    "nuestro horario de atencion. Gracias por su paciencia.",
    "Estimado/a, confirmamos la recepcion de su solicitud de documentos. Procedemos a prepararlos y "
    "se los remitiremos en breve.",
]

recuperador = None  # se inicializa solo si USE_RAG = True

if USE_RAG:
    try:
        from langchain_chroma import Chroma
        from langchain_huggingface import HuggingFaceEmbeddings

        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        vector_store = Chroma.from_texts(
            texts=PLANTILLAS,
            embedding=embeddings,
            collection_name="plantillas_respuesta",
        )
        recuperador = vector_store.as_retriever(search_kwargs={"k": 1})
        print("RAG activado: vector store de plantillas listo.")
    except ImportError:
        print(
            "RAG solicitado pero faltan dependencias "
            "(langchain-chroma, langchain-huggingface, sentence-transformers). "
            "Continuo SIN RAG."
        )
        USE_RAG = False
else:
    print("RAG desactivado (USE_RAG = False). Se usara borrador_respuesta del LLM.")


def recuperar_plantilla(texto: str) -> str:
    """Devuelve la plantilla mas parecida al texto, o cadena vacia si no hay RAG."""
    if not USE_RAG or recuperador is None:
        return ""
    docs = recuperador.invoke(texto)
    return docs[0].page_content if docs else ""


## 8. Funciones auxiliares

- `extraer_campos`: normaliza un mensaje de Gmail a `remitente / asunto / cuerpo`.
- `hilo_tiene_borrador`: **deduplicacion** — revisa si algun mensaje del hilo tiene la etiqueta `DRAFT`
  (equivale al IF de n8n). Usa `api_resource` porque la tool de hilo no expone las etiquetas.
- `redactar_borrador`: produce el texto del borrador; si `USE_RAG`, lo guia con la plantilla recuperada;
  si no, usa el `borrador_respuesta` del LLM.
- `crear_borrador_en_hilo`: crea el borrador **dentro del hilo** (`threadId`) via `api_resource`, para que
  quede asociado al correo original igual que en n8n (la tool `create_gmail_draft` no expone `threadId`).

In [ ]:
import base64
from email.mime.text import MIMEText
from langchain_core.output_parsers import StrOutputParser


def extraer_campos(msg: dict) -> dict:
    """Normaliza un mensaje devuelto por GmailSearch a remitente/asunto/cuerpo."""
    return {
        "remitente": msg.get("sender", "") or "",
        "asunto": msg.get("subject", "") or "",
        "cuerpo": (msg.get("body") or msg.get("snippet") or "").strip(),
    }


def hilo_tiene_borrador(thread_id: str) -> bool:
    """Deduplicacion: True si algun mensaje del hilo tiene la etiqueta DRAFT."""
    hilo = api_resource.users().threads().get(userId="me", id=thread_id).execute()
    for mensaje in hilo.get("messages", []):
        if "DRAFT" in mensaje.get("labelIds", []):
            return True
    return False


# Cadena auxiliar para redactar el borrador guiado por una plantilla (solo si USE_RAG).
prompt_borrador = ChatPromptTemplate.from_messages([
    ("system",
     "Eres un asistente que redacta respuestas de correo en espanol: breves, cordiales y "
     "profesionales. Usa la siguiente plantilla como guia de tono y estructura (adaptala al "
     "caso, no la copies literalmente):\n\n{plantilla}"),
    ("human",
     "Correo original:\nDe: {remitente}\nAsunto: {asunto}\nCuerpo: {cuerpo}\n\n"
     "Redacta unicamente el cuerpo de la respuesta."),
])
cadena_borrador = prompt_borrador | llm | StrOutputParser()


def redactar_borrador(triage, campos: dict) -> str:
    """Texto del borrador. Con RAG: guiado por plantilla. Sin RAG: el del LLM."""
    if USE_RAG:
        consulta = campos["asunto"] + " " + campos["cuerpo"]
        plantilla = recuperar_plantilla(consulta)
        return cadena_borrador.invoke({
            "plantilla": plantilla,
            "remitente": campos["remitente"],
            "asunto": campos["asunto"],
            "cuerpo": campos["cuerpo"],
        }).strip()
    return triage.borrador_respuesta


def crear_borrador_en_hilo(to: str, asunto: str, cuerpo: str, thread_id: str) -> dict:
    """Crea un borrador DENTRO del hilo (threadId), dirigido al remitente original."""
    asunto_resp = asunto if asunto.lower().startswith("re:") else "Re: " + asunto
    mime = MIMEText(cuerpo, "plain", "utf-8")
    mime["To"] = to
    mime["Subject"] = asunto_resp
    raw = base64.urlsafe_b64encode(mime.as_bytes()).decode()
    return api_resource.users().drafts().create(
        userId="me",
        body={"message": {"raw": raw, "threadId": thread_id}},
    ).execute()


## 9. Orquestacion principal (equivalencia con n8n)

Reproduce el flujo del Nivel 1:

1. Buscar no leidos (`GmailSearch`, `is:unread`, max 3).
2. Para cada correo: extraer campos -> clasificar -> **enrutar por categoria**:
   - `Urgente` / `Solicitud de informacion` y `requiere_respuesta` -> leer hilo -> **deduplicar** ->
     si no hay borrador, **crear borrador en el hilo**.
   - `Spam o promocion` / `Otro` -> registrar la decision.
3. Imprimir un resumen por correo (categoria, urgencia, accion).

`crear_borradores=False` permite una **pasada en seco** (clasifica y enruta, pero no escribe en Gmail),
util para una primera prueba.

In [ ]:
CATEGORIAS_RESPONDER = {"Urgente", "Solicitud de informacion"}


def procesar_bandeja(max_correos: int = 3, crear_borradores: bool = True):
    """Procesa los correos no leidos: clasifica, enruta, deduplica y crea borradores."""
    # 1. Buscar no leidos (Tool GmailSearch).
    correos = tool_buscar.invoke({
        "query": "is:unread",
        "resource": "messages",
        "max_results": max_correos,
    })

    if not correos:
        print("No hay correos no leidos.")
        return

    resumen = []
    for msg in correos:
        campos = extraer_campos(msg)
        thread_id = msg.get("threadId")

        # 2b. Clasificar (Chain).
        triage = cadena_clasificacion.invoke(campos)

        # 2c. Enrutar por categoria.
        if triage.categoria in CATEGORIAS_RESPONDER and triage.requiere_respuesta:
            if hilo_tiene_borrador(thread_id):
                accion = "OMITIDO (el hilo ya tiene un borrador)"
            elif not crear_borradores:
                accion = "SIMULADO (borrador no creado: pasada en seco)"
            else:
                cuerpo = redactar_borrador(triage, campos)
                crear_borrador_en_hilo(
                    to=campos["remitente"],
                    asunto=campos["asunto"],
                    cuerpo=cuerpo,
                    thread_id=thread_id,
                )
                accion = "BORRADOR creado en el hilo"
        else:
            # Spam o promocion / Otro -> solo registrar (etiquetado opcional via API).
            accion = "REGISTRADO (sin borrador)"

        resumen.append((campos["asunto"], triage.categoria, triage.urgencia, accion))

    # 3. Resumen final.
    print("\n" + "=" * 78)
    print("RESUMEN DEL TRIAGE")
    print("=" * 78)
    for asunto, categoria, urgencia, accion in resumen:
        print("- [" + categoria + " / " + urgencia + "] " + repr(asunto[:45]))
        print("    -> " + accion)


## 10. Ejecucion sobre la bandeja real

Procesa hasta 3 correos no leidos de tu cuenta.

> Primera vez: considera `crear_borradores=False` para una pasada en seco y verificar la clasificacion
> antes de escribir borradores reales en Gmail.

In [ ]:
# Pasada en seco (no escribe en Gmail):
# procesar_bandeja(max_correos=3, crear_borradores=False)

# Ejecucion real (crea borradores para Urgente / Solicitud de informacion):
procesar_bandeja(max_correos=3, crear_borradores=True)


## 11. Resumen de componentes (para la sustentacion)

| Componente | Donde, en este notebook |
|---|---|
| **LLM** (`ChatGroq`) | Seccion 2 |
| **Salida estructurada** (Pydantic) | Seccion 3 (`Triage`) |
| **CoT** (Zero-shot) | campo `razonamiento` de `Triage` |
| **ChatPromptTemplate** | Seccion 4 (`prompt_triage`) |
| **Chain** (LCEL) | Seccion 5 (`cadena_clasificacion`) |
| **Tools** (`GmailToolkit`) | Seccion 6 |
| **RAG** (opcional) | Seccion 7 (`USE_RAG`) |
| **Deduplicacion** | `hilo_tiene_borrador` (label `DRAFT`) |
| **Enrutamiento** (Switch de n8n) | `procesar_bandeja` (`if/elif`) |
| **Borrador en el hilo** | `crear_borrador_en_hilo` (`threadId`) |

**Equivalencia con n8n:** Gmail Trigger -> `GmailSearch`; nodo Groq -> `ChatGroq`; parser ->
`with_structured_output`; Switch -> `if/elif`; Gmail draft create -> `drafts().create()` con `threadId`;
thread get + IF -> `hilo_tiene_borrador`.
